# 05. 음성 → 텍스트: Whisper STT

**실습 목표**  
음성 파일을 자동으로 받아 적고, 다국어 음성 인식 파이프라인을 체험한다.

**주요 Hugging Face 모델**  
`openai/whisper-small`

> 이 노트북은 **파인튜닝 없이 사전학습 모델을 추론에 활용**하는 실습이다.  
> RTX 4060 8GB 환경을 고려했으며, CUDA가 없으면 CPU로 자동 전환하도록 구성하였다.

In [2]:
# uv add transformers accelerate torch torchaudio librosa soundfile

In [ ]:
import torch
print("PyTorch:", torch.__version__)

# 컴퓨터에 그래픽카드(GPU)가 있으면 계산이 훨씬 빨라져요. GPU가 있는지 확인!
print("CUDA available:", torch.cuda.is_available())

# transformers의 pipeline 기능은 GPU 번호를 숫자로 받아요 (0번 GPU, 없으면 -1 = CPU 사용)
DEVICE = 0 if torch.cuda.is_available() else -1
# torch 자체는 "cuda"(GPU) 또는 "cpu"라는 글자로 표시해요. 표기 방식만 다를 뿐 같은 의미예요
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", TORCH_DEVICE)

In [ ]:
from transformers import pipeline
import torch

# GPU가 있으면 float16(더 가볍고 빠른 계산 방식)을, 없으면 float32(기본 방식)를 사용해요
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
# 음성을 듣고 글자로 바꿔주는 "자동 음성 인식(ASR)" 파이프라인을 만들어요
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",  # OpenAI가 만든 음성 인식 모델
    device=DEVICE,
    dtype=dtype
)

In [ ]:
# sudo apt update
# sudo apt install -y ffmpeg

In [ ]:
# 영어
# AUDIO_URL = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"

# 한글  Bingsu/KSS_Dataset
# 인터넷에 있는 한국어 음성 파일 주소예요
AUDIO_URL = "https://datasets-server.huggingface.co/assets/Bingsu/KSS_Dataset/--/default/train/0/audio/audio.wav?Expires=1788812452&Signature=tgokuJQuLK~6aGF4z6~RTa9~p-s9Kj19~Sjn5ZfGCQdS4vFmtcIsuXHVIyHOh2SKJKjguKCKe9mkHKkKxMlKmV0z-rRrtQWHLp23fyq0xVf0QwxUUHE0M90K~sUaLs-HciW4mW0MuIjLv1o3kE34sKYOXdVQd9-ZZUubhVKpCwazlXTDVPxV3jHLBBWHOByC6eBYUJxvLbVR4ZI5HtjOcRALoJBBYRjpnoRr~IjVfDFnTjBNywDiC2CcI1D1ILFODtT-t2CULmlxY8oT~G~RQpcLeA1UceVw-U72io3jWReq4UWmcgTCWVJBj~fEgBnyIxRi1JkyQJP4XIZ8IjBDiA__&Key-Pair-Id=K3C0L9WB6U5DUC"
# 음성 파일 주소를 넣으면 asr 파이프라인이 알아서 다운로드해서 듣고 글자로 바꿔줘요
result = asr(AUDIO_URL)
print(result["text"])

In [ ]:
# 본인 음성 파일 사용 예
# 인터넷 주소 대신 내 컴퓨터에 있는 음성 파일 경로를 넣어도 똑같이 작동해요
result = asr("audio/audio_ko.wav")
# result = asr("audio/audio_ko.wav", return_timestamps=True)  # 몇 초에 무슨 말을 했는지도 알고 싶다면 이렇게!
result
# print(result["text"])

## 확장 과제
- 한국어 음성 파일을 녹음해 인식한다.
- `return_timestamps=True`를 적용해 자막용 타임스탬프를 생성해본다.

### 1. 마이크로 한국어 음성 녹음 후 인식
`sounddevice`로 마이크 입력을 몇 초간 녹음해 wav 파일로 저장하고, 그 파일을 Whisper로 인식해본다.
(로컬 컴퓨터에 마이크가 연결되어 있어야 하며, `uv add sounddevice` 설치가 필요하다.)

In [ ]:
import os
import sounddevice as sd
import scipy.io.wavfile as wavfile

SAMPLE_RATE = 16000  # Whisper 모델이 학습된 샘플링 레이트에 맞춰요
DURATION = 5  # 녹음 시간(초)

os.makedirs("audio", exist_ok=True)
RECORDED_PATH = "audio/my_recording.wav"

print(f"{DURATION}초 동안 한국어로 말해보세요...")
# 마이크로부터 오디오를 녹음해요 (channels=1은 모노 녹음)
recording = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32"
)
sd.wait()  # 녹음이 끝날 때까지 여기서 기다려요
print("녹음 완료!")

# 녹음한 데이터를 wav 파일로 저장해요
wavfile.write(RECORDED_PATH, SAMPLE_RATE, recording)
print("저장 위치:", RECORDED_PATH)

In [ ]:
# 방금 녹음한 파일을 Whisper로 인식해봐요
result = asr(RECORDED_PATH)
print("인식 결과:", result["text"])

### 2. `return_timestamps=True`로 자막용 타임스탬프 생성
`return_timestamps=True`를 넘기면 텍스트뿐 아니라 각 구간이 몇 초에서 몇 초까지인지도 함께 알려준다. 이를 이용해 자막(.srt) 파일을 만들어본다.

In [ ]:
# return_timestamps=True를 주면 결과에 "chunks"가 추가돼요. 각 chunk에는 텍스트와 (시작, 끝) 시간이 들어있어요
timestamped_result = asr("audio/audio_ko.wav", return_timestamps=True)

print("전체 텍스트:", timestamped_result["text"])
print("\n[구간별 타임스탬프]")
for chunk in timestamped_result["chunks"]:
    start, end = chunk["timestamp"]
    print(f"[{start:.2f}s -> {end:.2f}s] {chunk['text']}")

In [ ]:
# 초 단위 숫자를 자막 파일(.srt)에서 쓰는 "시:분:초,밀리초" 형식으로 바꿔주는 함수예요
def seconds_to_srt_time(seconds):
    if seconds is None:
        seconds = 0
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

# chunk마다 번호, 시작~끝 시간, 텍스트를 SRT 형식 줄로 만들어요
srt_lines = []
for i, chunk in enumerate(timestamped_result["chunks"], start=1):
    start, end = chunk["timestamp"]
    srt_lines.append(str(i))
    srt_lines.append(f"{seconds_to_srt_time(start)} --> {seconds_to_srt_time(end)}")
    srt_lines.append(chunk["text"].strip())
    srt_lines.append("")  # 자막 블록 사이 빈 줄

os.makedirs("output", exist_ok=True)
with open("output/subtitle.srt", "w", encoding="utf-8") as f:
    f.write("\n".join(srt_lines))

print("자막 파일 저장 완료: output/subtitle.srt")